## ÉTAPE 11 — Régression linéaire
Objectif : prédire la production de blé future


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv("JEU_DE_DONNEE.csv")


## 1. PRÉPARER LES DONNÉES


In [ ]:
df_ble = df[
    (df["category"] == "wheat") &
    (df["country_or_area"] == "World +") &
    (df["element"] == "Production Quantity")
].dropna(subset=["value"]).sort_values("year")

X = df_ble[["year"]].values          # Variable explicative (année)
y = df_ble["value"].values / 1e6     # Variable cible (production en Mt)

print(f"Données : {len(X)} années (1961 → 2007)")


## 2. SÉPARER TRAIN / TEST


## On entraîne le modèle sur 80% des données
et on teste sur les 20% restants


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train : {len(X_train)} exemples | Test : {len(X_test)} exemples")


## 3. ENTRAÎNER LE MODÈLE


In [ ]:
modele = LinearRegression()
modele.fit(X_train, y_train)   # Le modèle apprend la relation année → production

print(f"\nCoefficient (pente)    : {modele.coef_[0]:.2f} Mt/an")
print(f"Intercept (ordonnée)   : {modele.intercept_:.2f}")
print(f"→ Chaque année, la production augmente de {modele.coef_[0]:.1f} millions de tonnes")


## 4. ÉVALUER LE MODÈLE


In [ ]:
y_pred = modele.predict(X_test)
r2  = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print(f"\n--- Performance du modèle ---")
print(f"R² = {r2:.3f}  (1.0 = parfait, 0 = nul)")
print(f"MAE = {mae:.1f} Mt  (erreur moyenne de prédiction)")


## 5. PRÉDIRE LE FUTUR


In [ ]:
annees_futures = np.array([[2008], [2010], [2015], [2020], [2025]])
predictions = modele.predict(annees_futures)

print(f"\n--- Prédictions ---")
for annee, pred in zip(annees_futures.flatten(), predictions):
    print(f"  {annee} → {pred:.0f} Mt prévus")


## 6. GRAPHIQUE


In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

# Données réelles
ax.scatter(X, y, color="#2563EB", s=40, alpha=0.7, label="Données réelles", zorder=4)

# Droite de régression
x_line = np.linspace(1961, 2025, 100).reshape(-1, 1)
y_line = modele.predict(x_line)
ax.plot(x_line, y_line, color="red", linewidth=2.5,
        linestyle="--", label=f"Régression linéaire (R²={r2:.2f})")

# Prédictions futures
ax.scatter(annees_futures, predictions, color="orange",
           s=120, zorder=5, marker="*", label="Prédictions futures")
for annee, pred in zip(annees_futures.flatten(), predictions):
    ax.annotate(f"{pred:.0f} Mt", xy=(annee, pred),
                xytext=(annee + 0.3, pred + 5),
                fontsize=9, color="darkorange")

# Zone de séparation train/test
ax.axvline(x=2003, color="gray", linestyle=":", alpha=0.7)
ax.text(2003.3, y.min() + 10, "→ Futur prédit", fontsize=9, color="gray")

ax.set_title("Régression linéaire — Production mondiale de blé",
             fontsize=14, fontweight="bold")
ax.set_xlabel("Année")
ax.set_ylabel("Production (millions de tonnes)")
ax.legend(fontsize=11)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("regression_ble.png", dpi=150, bbox_inches="tight")
print("\nGraphique régression sauvegardé ✓")
plt.show()


## 7. RÉSUMÉ


In [ ]:
print("\n=== RÉSUMÉ RÉGRESSION ===")
print("LinearRegression().fit(X, y)  → entraîner")
print("modele.predict(X_new)          → prédire")
print("r2_score(y_test, y_pred)       → évaluer (R²)")
print("mean_absolute_error()          → erreur moyenne")
